# 15 — UCI Multicenter Real-only Baseline (920 rows)

Notebook Colab tái lập baseline trên bốn cohort UCI thật: Cleveland, Hungarian, Switzerland và Long Beach VA.

Mục tiêu của baseline này là tạo mốc so sánh sạch cho các thí nghiệm class weighting, threshold tuning và synthetic augmentation sau này. Không dùng dữ liệu synthetic, SMOTE hoặc H03 corruption trong notebook này.

> Đây là nghiên cứu trên dữ liệu công khai, không phải bằng chứng lâm sàng hay công cụ chẩn đoán.

In [ ]:
!pip -q install lightgbm seaborn

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
THRESHOLD = 0.50
OUTPUT_DIR = Path('/content/uci_multicenter_baseline_920_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 100)
print('Output directory:', OUTPUT_DIR)

## 1. Tải và kiểm tra bốn cohort UCI

Ký hiệu `?` được giữ là missing. `site` chỉ dùng để audit và chia LOCO, không đưa vào model. Target nhị phân được tạo bằng `num > 0`.

In [ ]:
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'
]
FEATURES = COLUMNS[:-1]
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {
    'cleveland': 'processed.cleveland.data',
    'hungarian': 'processed.hungarian.data',
    'switzerland': 'processed.switzerland.data',
    'va': 'processed.va.data',
}
EXPECTED_ROWS = {'cleveland': 303, 'hungarian': 294, 'switzerland': 123, 'va': 200}

frames = []
for site, filename in FILES.items():
    url = f'{BASE_URL}/{filename}'
    part = pd.read_csv(url, names=COLUMNS, na_values=['?'], skipinitialspace=True)
    part = part.apply(pd.to_numeric, errors='coerce')
    part['site'] = site
    part['target'] = (part['num'] > 0).astype('int8')
    assert len(part) == EXPECTED_ROWS[site], f'{site}: expected {EXPECTED_ROWS[site]}, got {len(part)}'
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
assert len(df) == 920, f'Expected 920 rows, got {len(df)}'
assert df[FEATURES].shape[1] == 13
assert df['target'].isin([0, 1]).all()

df[FEATURES + ['num', 'target', 'site']].to_csv(OUTPUT_DIR / 'uci_multicenter_real_only_raw.csv', index=False)
print('Combined shape:', df.shape)
display(df.head())

In [ ]:
site_summary = df.groupby('site').agg(
    rows=('target', 'size'),
    disease_count=('target', 'sum'),
    positive_rate=('target', 'mean'),
    age_mean=('age', 'mean'),
).reset_index()
site_missing = df.groupby('site')[FEATURES].apply(lambda x: x.isna().mean()).T
site_summary['missing_cells'] = site_summary['site'].map(
    df.groupby('site')[FEATURES].apply(lambda x: int(x.isna().sum().sum()))
)
site_summary['missing_rate'] = site_summary['missing_cells'] / (site_summary['rows'] * len(FEATURES))

print('Rows by site:')
display(site_summary.round(4))
print('Missing rate by feature and site (%):')
display((site_missing * 100).round(1))
print('Duplicate rows excluding site:', int(df[COLUMNS].duplicated().sum()))
print('Target distribution:')
display(pd.crosstab(df['site'], df['target'], margins=True))

site_summary.to_csv(OUTPUT_DIR / 'site_summary.csv', index=False)
site_missing.to_csv(OUTPUT_DIR / 'missing_by_site.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=site_summary, x='site', y='positive_rate', ax=axes[0])
axes[0].set_title('Disease prevalence by site')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=20)
sns.heatmap(site_missing * 100, annot=True, fmt='.1f', cmap='Reds', ax=axes[1])
axes[1].set_title('Missing rate (%) by feature and site')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'data_quality_by_site.png', dpi=180, bbox_inches='tight')
plt.show()

## 2. Leakage-safe baseline pipeline

Preprocessing được fit lại từ đầu trong từng LOCO fold:

- Numerical: median imputation + missing indicator + StandardScaler.
- Categorical: most-frequent imputation + missing indicator + OneHotEncoder.
- Model: Logistic Regression và LightGBM với `class_weight='balanced'`.
- Threshold cố định `0.50`; threshold tuning là thí nghiệm tiếp theo, không trộn vào baseline.

In [ ]:
def make_preprocessor():
    numeric = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ])
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ])
    return ColumnTransformer([
        ('numerical', numeric, NUMERICAL_FEATURES),
        ('categorical', categorical, CATEGORICAL_FEATURES),
    ])

def make_models():
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=250, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.9, colsample_bytree=0.9,
            reg_lambda=1.0, class_weight='balanced',
            random_state=RANDOM_STATE, verbosity=-1
        ),
    }

def metric_record(model_name, test_site, y_true, probability, fit_seconds):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        'model': model_name,
        'test_site': test_site,
        'test_rows': int(len(y_true)),
        'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability) if len(np.unique(y_true)) == 2 else np.nan,
        'brier': brier_score_loss(y_true, probability),
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp),
        'fit_seconds': fit_seconds,
    }

## 3. Leave-One-Center-Out evaluation

Mỗi site được khóa hoàn toàn làm test một lần. Không dùng test site để impute, chọn threshold, tune hyperparameter hay sinh dữ liệu.

In [ ]:
records = []

for test_site in FILES:
    train_df = df[df['site'] != test_site].copy()
    test_df = df[df['site'] == test_site].copy()
    X_train, y_train = train_df[FEATURES], train_df['target']
    X_test, y_test = test_df[FEATURES], test_df['target']

    for model_name, classifier in make_models().items():
        pipeline = Pipeline([
            ('preprocessor', make_preprocessor()),
            ('classifier', classifier),
        ])
        started = time.perf_counter()
        pipeline.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - started
        probability = pipeline.predict_proba(X_test)[:, 1]
        records.append(metric_record(model_name, test_site, y_test, probability, fit_seconds))

loco_results = pd.DataFrame(records).sort_values(['model', 'test_site']).reset_index(drop=True)
loco_results.to_csv(OUTPUT_DIR / 'multicenter_real_only_loco.csv', index=False)
display(loco_results.round(4))

In [ ]:
model_summary = loco_results.groupby('model').agg(
    roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'),
    roc_auc_worst=('roc_auc', 'min'),
    recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'),
    recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'),
    f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'),
    false_negatives_total=('false_negatives', 'sum'),
    fit_seconds_mean=('fit_seconds', 'mean'),
).sort_values(['roc_auc_worst', 'recall_worst'], ascending=False)

print('MULTICENTER REAL-ONLY BASELINE (920 rows)')
display(model_summary.round(6))
model_summary.round(6).to_csv(OUTPUT_DIR / 'multicenter_real_only_summary.csv')
model_summary.round(6).to_json(OUTPUT_DIR / 'multicenter_real_only_summary.json', orient='index', indent=2)

plot_data = loco_results.pivot(index='test_site', columns='model', values='roc_auc')
ax = plot_data.plot(kind='bar', figsize=(10, 4), ylim=(0.5, 1.0), rot=0)
ax.set_ylabel('ROC-AUC')
ax.set_title('Real-only external performance by held-out hospital')
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multicenter_real_only_roc_auc_by_site.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
winner = model_summary.reset_index().iloc[0]
print('MULTICENTER ROBUSTNESS CANDIDATE — REAL-ONLY')
display(winner)
print('Interpretation: empirical LOCO winner on four UCI cohorts; not clinical validation.')
print('Next: tune threshold and calibration inside training hospitals, then compare augmentation on the same held-out real sites.')

run_config = {
    'dataset': 'UCI Heart Disease, four processed cohorts',
    'expected_rows': 920,
    'features': FEATURES,
    'target_rule': 'target = (num > 0)',
    'validation': 'Leave-One-Center-Out',
    'threshold': THRESHOLD,
    'random_state': RANDOM_STATE,
    'synthetic_data': False,
    'smote': False,
    'source_urls': {site: f'{BASE_URL}/{filename}' for site, filename in FILES.items()},
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print('Saved artifacts to:', OUTPUT_DIR)

## 4. Tải artifact về máy

ZIP gồm raw data, audit, kết quả từng site, summary, config và hình minh họa. Hãy tải ZIP về repo để baseline Colab có thể tái lập.

In [ ]:
zip_path = shutil.make_archive('/content/uci_multicenter_baseline_920_real_only', 'zip', OUTPUT_DIR)
print('ZIP:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Không chạy trong Colab; ZIP vẫn nằm tại:', zip_path)

## Baseline checklist

- [x] 920 bệnh nhân thật từ 4 cohort.
- [x] Site chỉ dùng cho LOCO, không dùng làm feature.
- [x] Imputation và scaling fit trong training fold.
- [x] Không synthetic, không SMOTE, không H03 corruption.
- [x] Threshold 0.50 cố định để tạo mốc so sánh.
- [ ] Threshold tuning nested.
- [ ] Calibration và external validation trên nguồn dữ liệu mới.